# Playbook 02 — Disease Signature Generation

Differential expression (disease vs control) → up/down gene lists → ranked genes → cell-type-stratified signatures → JSON export.

**Inputs:** preprocessed AnnData (from `01_single_cell_preprocessing.ipynb`) with `obs['condition']` and `obs['cell_type']`.

**Output:** `artifacts/signatures/disease_signature.json` matching the gap-doc schema.

In [ ]:
import logging, json
from pathlib import Path
logging.basicConfig(level=logging.INFO)
import numpy as np
import pandas as pd

## 1. Build a small AnnData (substitute the output of playbook 01 in production)

In [ ]:
import anndata as ad, scanpy as sc
rng = np.random.default_rng(7)
n_cells, n_genes = 600, 1500
X = rng.negative_binomial(5, 0.3, size=(n_cells, n_genes)).astype('float32')
adata = ad.AnnData(X=X)
adata.var_names = [f'GENE_{i:04d}' for i in range(n_genes)]
adata.obs['condition'] = ['disease']*300 + ['control']*300
adata.obs['cell_type'] = (['T_cell']*200 + ['B_cell']*200 + ['NK_cell']*200)
# Inject a synthetic disease signal in the first 30 genes for disease cells
adata.X[:300, :30] += rng.normal(5, 1, size=(300, 30))
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(adata)

## 2. Run differential expression (Wilcoxon, disease vs control)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='condition', reference='control', method='wilcoxon')
names = pd.DataFrame(adata.uns['rank_genes_groups']['names'])['disease']
scores = pd.DataFrame(adata.uns['rank_genes_groups']['scores'])['disease']
lfc = pd.DataFrame(adata.uns['rank_genes_groups']['logfoldchanges'])['disease']
padj = pd.DataFrame(adata.uns['rank_genes_groups']['pvals_adj'])['disease']
de_df = pd.DataFrame({
    'names': names, 'scores': scores, 'logfoldchanges': lfc, 'pvals_adj': padj,
})
de_df.head()

## 3. Build disease signature (single cohort)

In [ ]:
from single_cell_layer.disease_signature import build_disease_signature
sig = build_disease_signature(
    de_df,
    disease_id='Disease::DOID:9352',
    tissue='whole_blood',
    cell_type='all_cells',
)
print(f'Up: {len(sig["up_genes"])} genes; Down: {len(sig["down_genes"])} genes')
print('Top-5 up:', sig['up_genes'][:5])

## 4. Cell-type-stratified signatures + consensus

In [ ]:
from single_cell_layer.cell_type_signature import build_per_cell_type_signatures, consensus_signature
# Add cell_type to de_df so stratifier has something to split on
de_df['cell_type'] = np.tile(['T_cell','B_cell','NK_cell'], len(de_df)//3 + 1)[:len(de_df)]
per_ct = build_per_cell_type_signatures(de_df, disease_id='Disease::DOID:9352', tissue='whole_blood')
consensus_up, consensus_down = consensus_signature(per_ct, min_cell_types=2)
print(f'Per-cell-type signatures: {list(per_ct.keys())}')
print(f'Consensus: {len(consensus_up)} up, {len(consensus_down)} down')

## 5. Export

In [ ]:
out = Path('artifacts/signatures')
out.mkdir(parents=True, exist_ok=True)
(out / 'disease_signature.json').write_text(json.dumps(sig, indent=2))
(out / 'cell_type_signatures.json').write_text(json.dumps(per_ct, indent=2))
print(f'Wrote {out}/disease_signature.json and {out}/cell_type_signatures.json')